In [ ]:
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.curdir, '..')))
from src.utils import *
from src.panda_program import PandaMugProgram
from pydrake.all import (
    StartMeshcat,
    Quaternion,
    RigidTransform,
    RotationMatrix,
    MinimumDistanceLowerBoundConstraint,
    SceneGraphInspector,
)



ikflow/config.py | Using device: 'cuda:0'


In [ ]:
meshcat = StartMeshcat()
diagram = BuildEnv(meshcat=meshcat, directives_file = os.path.join(RepoDir(), "models/panda/panda_collision.yaml"))

program = PandaMugProgram(diagram)
pose = RigidTransform(
  R=RotationMatrix([
    [-0.17766567179807877, 0.8630694524675712, -0.4728065453035352],
    [0.4622397335466241, 0.49733973016466565, 0.7341577633795637],
    [0.868774618526165, -0.08811533928360074, -0.48730519101242153],
  ]),
  p=[0.13975740280531287, -0.14268883816803044, 0.9462502062891363],
)

target_mug = Mug(pose, height = 0.3)
program.create_prog(target_mug=target_mug)


# result = program.Solve()

# result.is_success()



INFO:drake:Meshcat listening for connections at http://localhost:7000


WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/panda_arm_hand_formatted_link_filepaths_absolute.urdf
joint mimic: no multiplier, using default value of 1 
joint mimic: no offset, using default value of 0 
URDFParser: Link size: 17
URDFParser: Joint size: 12
Geometry: Loading 12 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link0.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link3.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link4.dae into Group
Geometry: Loading 3 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link5.dae into Group
Geometry: Loading 17 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link6.dae into Group
Geometry: Loa

In [14]:
q = np.zeros(9)
q[:7] = np.random.uniform(-np.pi, np.pi, 7)
program.plant.SetPositions(program.plant_context, q)
print(program.collision_free_constraint.Eval(q))
program.diagram.ForcedPublish(program.diagram_context)
pose = program.frame.CalcPoseInWorld(program.plant_context)
print(pose)
print(*np.array([*pose.translation(), *pose.rotation().ToQuaternion().wxyz()]), sep = ", ")


[0.63035838]
RigidTransform(
  R=RotationMatrix([
    [-0.17766567179807877, 0.8630694524675712, -0.4728065453035352],
    [0.4622397335466241, 0.49733973016466565, 0.7341577633795637],
    [0.868774618526165, -0.08811533928360074, -0.48730519101242153],
  ]),
  p=[0.13975740280531287, -0.14268883816803044, 0.9462502062891363],
)
0.13975740280531287, -0.14268883816803044, 0.9462502062891363, 0.45617125823372673, -0.4506383774851174, -0.7352398576272858, -0.21967063448546764


In [4]:
pose = program.target_pose
pose = RigidTransform(Quaternion(pose[3], pose[4], pose[5], pose[6]), [pose[0], pose[1], pose[2]])
DrawAxes(pose, meshcat)

RuntimeError: Quaternion is not normalized

In [ ]:
result = program.Solve()

print(result.is_success())

vars = result.get_x_val()
print(vars[:7])
print(vars[7:-7])
print(vars[-7:])

True
[ 0.10722592 -0.01801715  0.99749323  0.37726557 -0.43033868 -0.73833421
 -0.12332272]
[-0.37499568 -0.97774935 -0.00812827 -0.31385295 -0.66233298  0.30775326
  0.40599587]
[-0.02007788  0.08399244  0.26065708 -0.3027391  -0.23676986 -0.20302109
  0.09949835]


In [ ]:
q = program.VarsToQ(vars)
program.plant.SetPositions(program.plant_context, q)
program.diagram.ForcedPublish(program.diagram_context)


pose = program.target_pose
pose = RigidTransform(Quaternion(pose[3], pose[4], pose[5], pose[6]), [pose[0], pose[1], pose[2]])
DrawAxes(pose, meshcat)


In [ ]:
print("Position Error", program.EvalPositionError(vars))
print("Orientation Error", program.EvalOrientationError(vars)) ### This is a lot


# joint limits

print("lower joint limits", program.plant.GetPositionLowerLimits()[:-2])
print("upper joint limits", program.plant.GetPositionUpperLimits()[:-2])
print("solution joint values", q[:-2])

Position Error [9.93430514e-06 3.40234886e-06 3.45908472e-06]
Orientation Error [0.00516336]
lower joint limits [-2.8973 -1.7628 -2.8973 -3.0718 -2.8973 -0.0175 -2.8973]
upper joint limits [ 2.8973  1.7628  2.8973 -0.0698  2.8973  3.7525  2.8973]
solution joint values [-2.41226983 -0.34970003  1.0939877  -1.05991244 -2.13686752  0.157001
  1.4495157 ]


In [ ]:
collision_free_constraint = MinimumDistanceLowerBoundConstraint(
            plant=program.plant,
            bound=1e-3,
            influence_distance_offset=1e-1,
            plant_context=program.plant_context
        )
collision_free_constraint.Eval(q)

array([0.98731893])